# Possession Play — Spielerdatenbank bauen (Kaggle)

Dieses Notebook erzeugt das vollständige `PLAYERS`-Array für das Spiel aus dem
Transfermarkt-Dataset **`davidcariboo/player-scores`** ("Football Data from
Transfermarkt"). Es führt die Logik der Repo-Skripte `data-pipeline/build_db.py`
und `data-pipeline/make_game_json.py` nacheinander aus.

Vereinszugehörigkeiten kommen aus zwei Quellen:
* **Einsätze** (`appearances.csv`) — präzise, reichen aber nur ~2012 zurück.
* **Transferhistorie** (`transfers.csv`) — ergänzt auch Stationen **vor 2012**.

Transfer-Vereine werden über die Klub-ID auf den **offiziellen Namen** aufgelöst;
**Jugend-/Reserve-/B-Teams** werden herausgefiltert (nur Profi-Stationen zählen).

## So benutzt du es

1. Auf https://www.kaggle.com → **Create → New Notebook** (Telefon-Verifizierung
   in den Settings ist nötig, sonst kein "Add Input").
2. Rechts **Add Input → Datasets** → **`davidcariboo/player-scores`** hinzufügen
   (Anzeigetitel: *Football Data from Transfermarkt*, Autor `davidcariboo`).
3. Dieses `.ipynb` hochladen (**File → Import Notebook**) oder den Inhalt einfügen.
4. Oben **Run All**.
5. Den **Vereins-Prüfbericht** kontrollieren: Jeder der 40 Spiel-Vereine sollte
   **genau einen** korrekten Profi-Namen matchen (keine Jugendteams, keine
   fremden Klubs). Bei Abweichung den Teilstring in `GAME_CLUBS` (Zelle 1)
   anpassen und erneut **Run All**.
6. Rechts unter **Output → /kaggle/working/** **`players_game.js`** herunterladen.

## Mount-Pfad prüfen (wichtig)

Der Datensatz mountet je nach Kaggle-Version unter
`/kaggle/input/player-scores/` **oder** `/kaggle/input/datasets/davidcariboo/player-scores/`.
`DATA` in Zelle 1 ist auf letzteren Pfad gesetzt. Stimmt der Pfad nicht, mit
`os.walk("/kaggle/input")` prüfen und `DATA` auf den Ordner mit `players.csv` setzen.

## Danach (im Repo)

Der Inhalt von `players_game.js` ersetzt `src/players.js` — siehe
`data-pipeline/README.md`. Die Ausgabe beginnt mit `export const` (1:1-Tausch).

> Caveat: Auch mit der Transferhistorie sind sehr alte / Leihstationen nicht
> garantiert lückenlos. "seit 2000" = `games.season >= 2000`.

In [ ]:
# ── Konfiguration & Helfer ─────────────────────────────────────────────
from pathlib import Path
import json, re, unicodedata
import pandas as pd

DATA = Path("/kaggle/input/datasets/davidcariboo/player-scores")
OUT  = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

TOP5 = {
    "GB1": ("Premier League", "England"),
    "ES1": ("LaLiga",         "Spain"),
    "IT1": ("Serie A",        "Italy"),
    "L1":  ("Bundesliga",     "Germany"),
    "FR1": ("Ligue 1",        "France"),
}
SINCE = 2000

# Spielkey -> Teilstring im (offiziellen) TM-Vereinsnamen, gegen die echte
# Klubliste geprüft. LIL/OL/ROM bewusst spezifisch (Lillestrøm/Lyon-Duchère/
# Cisco Roma ausschließen).
GAME_CLUBS = {
    "FCB": "bayern", "BVB": "dortmund", "RBL": "leipzig", "B04": "leverkusen",
    "SGE": "frankfurt", "BMG": "gladbach", "VFB": "stuttgart", "WOB": "wolfsburg", "SVW": "bremen",
    "MCI": "manchester city", "MUN": "manchester united", "LIV": "liverpool", "CHE": "chelsea",
    "ARS": "arsenal football club", "TOT": "tottenham", "NEW": "newcastle", "EVE": "everton", "AVL": "aston villa",
    "BAR": "futbol club barcelona", "RMA": "real madrid", "ATM": "atletico de madrid", "SEV": "sevilla",
    "VAL": "valencia", "VIL": "villarreal",
    "JUV": "juventus", "MIL": "calcio milan", "INT": "internazionale", "NAP": "napoli", "ROM": "sportiva roma", "LAZ": "lazio",
    "PSG": "saint-germain", "ASM": "monaco", "OM": "marseille", "OL": "olympique lyon", "LIL": "lille olympique",
    "POR": "clube do porto", "SLB": "benfica", "SCP": "clube de portugal",
    "AJA": "ajax", "PSV": "philips", "FEY": "feyenoord",
}

NATION_MAP = {
    "france": "FRA", "germany": "GER", "spain": "ESP", "italy": "ITA", "netherlands": "NED",
    "belgium": "BEL", "croatia": "CRO", "england": "ENG", "portugal": "PRT", "japan": "JPN",
    "brazil": "BRA", "argentina": "ARG", "mexico": "MEX", "nigeria": "NGA",
    "cote d'ivoire": "CIV", "ivory coast": "CIV", "senegal": "SEN", "colombia": "COL",
    "united states": "USA", "usa": "USA",
}

ONLY_GAME_CLUBS = True

def norm(s: str) -> str:
    s = unicodedata.normalize("NFD", str(s)).encode("ascii", "ignore").decode()
    return s.lower().strip()

# Jugend-/Reserve-/B-Team-Erkennung (ganze Tokens im normalisierten Namen)
_YOUTH = re.compile(r"(?:^| )(u\d{1,2}|sub-?\d{1,2}|ii|iii|b|c|res|reserves?|youth|yth|jugend|castilla|mestalla|amateure?)(?:$| )")
def is_youth(name) -> bool:
    return bool(_YOUTH.search(norm(name)))

def club_key_for(tm_name) -> "str|None":
    if is_youth(tm_name):
        return None
    n = norm(tm_name)
    for key, needle in GAME_CLUBS.items():
        if needle in n:
            return key
    return None

print("Konfiguration geladen. DATA =", DATA)

In [ ]:
# ── Schritt 1: Spielerauswahl (Top-5 seit 2000) + VOLLE Vereinshistorie ──
players      = pd.read_csv(DATA / "players.csv")
clubs        = pd.read_csv(DATA / "clubs.csv")
games        = pd.read_csv(DATA / "games.csv")
appearances  = pd.read_csv(DATA / "appearances.csv")

# Spiele seit 2000: einmal nur Top-5 (Spielerauswahl), einmal alle (Vereinshistorie)
g_top5 = games[games["competition_id"].isin(TOP5) & (games["season"] >= SINCE)][["game_id"]]
g_all  = games[games["season"] >= SINCE][["game_id", "season"]]

# Spielerauswahl: jeder Spieler mit >=1 Top-5-Einsatz seit 2000
app_top5 = appearances.merge(g_top5, on="game_id", how="inner")
pids = app_top5["player_id"].unique()

# VOLLE Vereinshistorie dieser Spieler: ALLE Einsätze seit 2000 (auch Portugal/NL/Pokale)
app = appearances[appearances["player_id"].isin(pids)].merge(g_all, on="game_id", how="inner")

# players.csv – nur die ausgewählten Spieler
pl = players[players["player_id"].isin(pids)].copy()
pl = pl[["player_id", "name", "first_name", "last_name",
         "date_of_birth", "position", "country_of_citizenship"]]
pl.columns = ["tm_player_id", "name", "first_name", "last_name",
              "date_of_birth", "position", "citizenship"]
pl["last_name"] = pl["last_name"].fillna("").where(
    pl["last_name"].notna() & (pl["last_name"] != ""), pl["name"])
pl.to_csv(OUT / "players.csv", index=False)

# clubs.csv – alle Vereine aus der vollen Historie
cids = app["player_club_id"].unique()
cl = clubs[clubs["club_id"].isin(cids)].copy()
cl = cl[["club_id", "name", "domestic_competition_id"]]
cl.columns = ["tm_club_id", "name", "tm_competition_id"]
cl.to_csv(OUT / "clubs.csv", index=False)

# player_club_spells.csv – (Spieler, Verein) -> Saison-Spanne
spells = (
    app.groupby(["player_id", "player_club_id"])["season"]
       .agg(["min", "max"])
       .reset_index()
)
spells.columns = ["tm_player_id", "tm_club_id", "season_start", "season_end"]
spells.to_csv(OUT / "player_club_spells.csv", index=False)

print(f"OK  players={len(pl):>6}  clubs={len(cl):>4}  spells={len(spells):>6}  -> {OUT}/")

In [ ]:
# ── Schritt 2: auf Spiel-Vereine mappen (Einsätze + Transferhistorie) ──
players     = pd.read_csv(OUT / "players.csv")
clubs       = pd.read_csv(OUT / "clubs.csv")
spells      = pd.read_csv(OUT / "player_club_spells.csv")
transfers   = pd.read_csv(DATA / "transfers.csv")     # Transferhistorie (auch vor 2012)
full_clubs  = pd.read_csv(DATA / "clubs.csv")          # für Klub-ID -> offizieller Name
id2name = dict(zip(full_clubs["club_id"], full_clubs["name"]))

pset = set(players["tm_player_id"])
tr = transfers[transfers["player_id"].isin(pset)].copy()
# Transfer-Vereine auf offizielle Namen auflösen (sauberer als die Kurznamen)
tr["from_name"] = tr["from_club_id"].map(id2name)
tr["to_name"]   = tr["to_club_id"].map(id2name)

# tm_club_id -> Spielkey (aus den Einsatz-Vereinen)
club_key = {c["tm_club_id"]: club_key_for(c["name"]) for _, c in clubs.iterrows()}

# PRÜFBERICHT über ALLE zu mappenden Namen (Einsätze + Transfers, ohne Jugend)
all_names = set(clubs["name"].dropna())
all_names |= set(tr["from_name"].dropna()) | set(tr["to_name"].dropna())
matched_names = {}
for nm in all_names:
    k = club_key_for(nm)
    if k:
        matched_names.setdefault(k, set()).add(nm)

print("── Vereins-Zuordnung (kontrollieren!) ──")
for k in GAME_CLUBS:
    names = sorted(matched_names.get(k, []))
    flag = "" if names else "  ⚠️ KEIN TREFFER"
    print(f"  {k}: {', '.join(names) if names else '—'}{flag}")
print("────────────────────────────────────────")

# Spieler -> Set der Spielkeys: (a) aus Einsätzen, (b) aus Transfers
pclubs = {}
for _, s in spells.iterrows():
    k = club_key.get(s["tm_club_id"])
    if k:
        pclubs.setdefault(s["tm_player_id"], set()).add(k)
for _, t in tr.iterrows():
    for nm in (t["from_name"], t["to_name"]):
        k = club_key_for(nm)
        if k:
            pclubs.setdefault(t["player_id"], set()).add(k)

out = []
for _, p in players.iterrows():
    keys = sorted(pclubs.get(p["tm_player_id"], []))
    if ONLY_GAME_CLUBS and not keys:
        continue  # für dieses Spiel irrelevant
    nat = NATION_MAP.get(norm(p.get("citizenship", "")))
    try:
        by = int(str(p["date_of_birth"])[:4])
    except Exception:
        continue
    out.append({
        "n": p["name"],
        "ln": p["last_name"] if isinstance(p["last_name"], str) and p["last_name"].strip() else p["name"],
        "by": by,
        "nat": [nat] if nat else [],
        "clubs": keys,
    })

out.sort(key=lambda r: r["n"])
body = ",\n  ".join(json.dumps(r, ensure_ascii=False) for r in out)
js = "export const PLAYERS = [\n  " + body + "\n];\n"
(OUT / "players_game.js").write_text(js, encoding="utf-8")
print(f"OK  {len(out)} Spieler -> {OUT}/players_game.js")
print("Download: rechts unter Output, dann Inhalt nach src/players.js übernehmen.")